# Part 4 (실습) — 출력 파서로 3단 체인 완성하기

> **이 노트북의 목표**
> 모델 출력(AIMessage)을 깔끔한 문자열·구조화 데이터로 받음. `StrOutputParser`, `JsonOutputParser`, Pydantic + `with_structured_output`을 직접 돌려 `프롬프트 | 모델 | 파서` 3단 체인을 완성함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~3

## 0. 준비

In [ ]:
%pip install -q -U langchain langchain-google-genai

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0)  # 추출은 0이 안정적
print("준비 완료")

## 1. StrOutputParser — 본문만 깔끔히

### 파서가 없으면? (Part 3 복습)
모델 출력은 AIMessage라, 본문을 보려면 매번 `.content`를 꺼내야 함.

In [ ]:
prompt = ChatPromptTemplate.from_template("{product}의 홍보 문구를 한 줄 써줘")

no_parser = prompt | model
r = no_parser.invoke({"product": "베이지 니트"})
print("타입:", type(r).__name__, "→ .content 필요:", r.content)

### StrOutputParser를 끼우면 결과가 바로 str
`(lambda m: m.content)`로 하던 일을 정식 블록으로.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()
r = chain.invoke({"product": "베이지 니트"})
print("타입:", type(r).__name__, "→", r)   # 이제 그냥 str

> 📌 결과가 str이라 이후 텍스트 가공(예: 해시태그 변환)을 자연스럽게 이을 수 있음. 스트리밍에서도 chunk가 바로 문자열이 됨.

In [ ]:
# str이라 그대로 다음 가공 블록을 이을 수 있음
chain2 = prompt | model | StrOutputParser() | (lambda s: "📢 " + s.strip())
print(chain2.invoke({"product": "린넨 셔츠"}))

## 2. JsonOutputParser — 딕셔너리로 받기

### 개념
모델 텍스트를 파이썬 딕셔너리/리스트로 변환함. 핵심은 **양방향**:
1. `get_format_instructions()`로 모델에게 형식을 지시
2. 모델 출력을 파서가 해석

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

# 파서가 만들어주는 형식 지시문을 프롬프트에 끼움
prompt_json = ChatPromptTemplate.from_template(
    "다음 상품 정보를 JSON으로 정리해줘. 키는 name, price_range, codi(리스트).\n"
    "상품: {product}\n{format_instructions}"
).partial(format_instructions=parser.get_format_instructions())

chain_json = prompt_json | model | parser
result = chain_json.invoke({"product": "베이지 케이블 니트"})

print("타입:", type(result).__name__)       # dict
print("name:", result["name"])
print("codi:", result["codi"])               # 키로 바로 접근

### format_instructions가 실제 무엇인지 들여다보기

In [ ]:
print(parser.get_format_instructions())

## 3. Pydantic + with_structured_output — 타입까지 보장

### 개념
출력의 구조·타입을 클래스로 미리 정의함. `with_structured_output`은 1.0 권장 방식 — 구조를 강제하고 **검증된 객체를 바로** 돌려줌(format_instructions 불필요).

In [ ]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(description="상품 이름")
    price_range: str = Field(description="예상 가격대 (예: 5만원대)")
    codi: list[str] = Field(description="추천 코디 3가지")

structured_model = model.with_structured_output(Product)

result = structured_model.invoke("베이지 케이블 니트를 정리해줘")
print("타입:", type(result).__name__)   # Product
print("name :", result.name)            # 점(.)으로 접근
print("price:", result.price_range)
print("codi :", result.codi)            # list[str] 보장

### dict vs 객체 — 무엇이 다른가
JsonOutputParser는 dict(키 접근, 타입 미보장), with_structured_output은 검증된 객체(점 접근, 타입 보장).

In [ ]:
# 객체이므로 IDE 자동완성·타입 검증이 됨. 잘못된 필드 접근은 즉시 에러로 드러남
print("코디 개수:", len(result.codi))
for i, c in enumerate(result.codi, 1):
    print(f"  {i}. {c}")

### 프롬프트와 함께 쓰기 (체인으로)
구조화 모델도 Runnable이라 프롬프트와 파이프로 이을 수 있음.

In [ ]:
structured_chain = (
    ChatPromptTemplate.from_template("리리마켓 상품 '{product}'의 정보를 정리해줘")
    | model.with_structured_output(Product)
)
p = structured_chain.invoke({"product": "캐시미어 머플러"})
print(p.name, "|", p.price_range, "|", p.codi)

## 4. 실전 — 고객 리뷰를 구조화 데이터로

리리마켓 고객 리뷰를 받아 감정·요약·키워드로 구조화함. (DB 저장·분기 판단에 바로 쓸 형태)

In [ ]:
from typing import Literal

class ReviewAnalysis(BaseModel):
    sentiment: Literal["긍정", "부정", "중립"] = Field(description="전반적 감정")
    summary: str = Field(description="리뷰 한 줄 요약")
    keywords: list[str] = Field(description="핵심 키워드 2~4개")

review_model = model.with_structured_output(ReviewAnalysis)

reviews = [
    "니트 질감이 정말 부드럽고 색도 사진이랑 똑같아요. 배송도 빨랐어요!",
    "사이즈가 너무 작게 나왔고 보풀이 금방 일어나요. 실망입니다.",
]
for rv in reviews:
    a = review_model.invoke(f"다음 리뷰를 분석해줘: {rv}")
    print(f"[{a.sentiment}] {a.summary}  / 키워드: {a.keywords}")

> 📌 `Literal["긍정","부정","중립"]`처럼 값의 범위를 못 박으면, 모델이 그 셋 중 하나만 답하도록 강제됨. 분기 판단에 안전함.

## ✏️ 미니 실습

**과제**: Pydantic 모델로 "영화 추천"을 구조화하기.
- 필드: `title`(str), `genre`(str), `reason`(str, 추천 이유)
- `with_structured_output`으로 "가을밤에 어울리는 영화 한 편 추천해줘" 호출

In [ ]:
# TODO: 직접 작성
# class Movie(BaseModel): ...
# rec = model.with_structured_output(Movie).invoke("...")
# print(rec.title, rec.genre, rec.reason)

<details><summary>👉 정답</summary>

```python
class Movie(BaseModel):
    title: str = Field(description="영화 제목")
    genre: str = Field(description="장르")
    reason: str = Field(description="추천 이유")

rec = model.with_structured_output(Movie).invoke("가을밤에 어울리는 영화 한 편 추천해줘")
print(rec.title, "|", rec.genre, "|", rec.reason)
```
</details>

## 정리

| 절 | 파서 | 결과 타입 | 쓰임 |
|---|---|---|---|
| 1 | `StrOutputParser` | `str` | 사람이 읽을 답 |
| 2 | `JsonOutputParser` | `dict`/`list` | 코드용 데이터 (형식 지시 필요) |
| 3 | `with_structured_output` | 검증된 객체 | 타입 보장, DB·분기·API |

### 3줄 요약
1. 파서는 체인의 마지막 블록 — 모델 출력을 원하는 형태로 변환함.
2. 사람용은 StrOutputParser, 코드용은 구조화 파서.
3. 1.0에선 `with_structured_output`이 가장 안정적인 구조화 방식임.

### ▶ 다음 (Part 5)
배운 모든 것(모델·프롬프트·LCEL·파서)을 합쳐 **첫 미니 프로젝트**를 만듦. 초급의 졸업 작품임.